# Intermediate RAG System Using Pinecone Vector Database

A Retrieval-Augmented Generation (RAG) system that answers questions strictly from an uploaded PDF, using Pinecone for vector storage, Sentence-Transformers for embeddings, and Groq's Llama 3 for answer generation.

**Author:** Sadaf
**Course:** Artificial Intelligence / NLP / Applied LLM Systems
**Level:** Intermediate

# New section

## 1. Project Objective

Design and implement a RAG system that:
- Accepts one or more PDF documents as input
- Extracts and processes text from the document(s)
- Generates embeddings for text chunks
- Stores embeddings in the Pinecone vector database
- Retrieves relevant context based on a user query
- Generates accurate answers strictly from the PDF content
- Prevents hallucination and provides traceable, source-attributed answers

## 2. What This Project Does

The system lets a user upload PDF document(s), then ask natural-language questions about their content. Instead of relying on the LLM's general knowledge, the system first retrieves the most relevant text chunks from the document (semantic search via Pinecone), and instructs the LLM to answer **only** using those chunks. If no relevant chunk is found, the system explicitly says the answer is not available in the document, rather than guessing.

## 3. How the System Works — Architecture

```
PDF Upload
    |
Text Extraction (pdfplumber, page-wise)
    |
Text Chunking (LangChain RecursiveCharacterTextSplitter, adjustable size/overlap)
    |
Embedding Generation (Sentence-Transformers: all-MiniLM-L6-v2, 384-dim)
    |
Pinecone Vector Indexing (namespace + metadata: page, doc name, chunk id)
    |
Semantic Retrieval (cosine similarity, adjustable top-k and threshold)
    |
LLM Response Generation (Groq Llama 3, context-restricted prompt)
    |
Answer with Source Reference (page number, excerpt, similarity score)
```

## 4. Complete Workflow (Step by Step)

1. User provides Pinecone and Groq API keys.
2. Pipeline connects to Pinecone and creates/loads a serverless index.
3. User uploads PDF(s); each is validated (format, size, readability).
4. Text is extracted page-by-page and cleaned of formatting artifacts.
5. Cleaned text is split into overlapping chunks (chunk size/overlap adjustable from the UI).
6. Each chunk is converted into a 384-dimension embedding vector.
7. Vectors are upserted into Pinecone with metadata (page number, document name, chunk ID) inside a namespace.
8. User asks a question; the question is embedded using the same model.
9. Pinecone is queried for the top-k most similar chunks (cosine similarity), filtered by a similarity threshold and optional metadata filters (page/document).
10. Retrieved chunks are passed to the LLM with a strict system prompt: answer only from context, or state that the answer is not available.
11. The final answer is displayed along with source page numbers, excerpts, and similarity scores.
12. The query and confidence score are logged for session history and auditing.

## 5. Why Each Library Is Used

| Library | Purpose |
|---|---|
| `pdfplumber` | Reliable page-wise text extraction from PDFs |
| `langchain` | `RecursiveCharacterTextSplitter` for intelligent, structure-aware chunking |
| `sentence-transformers` | Generates dense vector embeddings locally, free, fast on Colab |
| `pinecone` | Managed vector database: index creation, namespaces, upsert, similarity query, metadata filtering |
| `groq` | Fast, free-tier LLM inference (Llama 3) for answer generation |
| `streamlit` | Interactive web UI for upload, configuration, and Q&A |
| `pyngrok` | Exposes the Streamlit app running in Colab via a public URL |

## 6. Major Components (Modular Architecture)

- `pdf_loader.py` — PDF validation and text extraction
- `text_chunker.py` — Adjustable text chunking
- `embedder.py` — Embedding generation
- `vector_store.py` — All Pinecone operations (index, namespace, upsert, query)
- `retriever.py` — Top-k retrieval, thresholding, confidence scoring
- `generator.py` — LLM-based, hallucination-restricted answer generation
- `query_logger.py` — Query history and logging
- `rag_pipeline.py` — Orchestrates the full pipeline end-to-end
- `app.py` — Streamlit UI tying everything together

## 7. Folder Structure (created in this Colab session)

```
/content/
  pdf_loader.py
  text_chunker.py
  embedder.py
  vector_store.py
  retriever.py
  generator.py
  query_logger.py
  rag_pipeline.py
  app.py
  query_log.csv        (created after first query)
```

## 8. Expected Output

- A working Streamlit web app (public URL via ngrok) where you can upload PDFs and ask questions.
- Answers grounded strictly in the uploaded document, each with page number, excerpt, and similarity score.
- A fallback message — "The answer is not available in the provided document." — whenever the document does not contain the answer.
- A query log (`query_log.csv`) and in-session query history panel.


## 9. Install Required Packages

In [ ]:
!pip uninstall -y pillow -q
!pip install -q --no-cache-dir --force-reinstall "pillow==12.2.0"
!pip install -q pdfplumber langchain langchain-text-splitters sentence-transformers pinecone groq streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 68.5 MB/s eta 0:00:00


## 10. API Key Setup\n\nEnter your API keys below. Keys are requested at runtime via `getpass` — never hardcoded in the notebook.\n\n- Get a free Pinecone API key: https://app.pinecone.io\n- Get a free Groq API key: https://console.groq.com

In [ ]:
import os
from getpass import getpass

os.environ["PINECONE_API_KEY"] = getpass("Enter your Pinecone API key: ")
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

print("API keys stored in environment variables for this session.")

## 11. Create Modular Project Files\n\nEach module below is written to its own `.py` file, keeping the architecture clean and separated by responsibility (loader, chunker, embedder, vector store, retriever, generator, logger, pipeline).

In [ ]:
%%writefile pdf_loader.py
"""
pdf_loader.py
Handles PDF validation and page-wise text extraction with cleanup.
"""

import os
import re
import pdfplumber


MAX_FILE_SIZE_MB = 20


class PDFLoadError(Exception):
    pass


def validate_pdf(file_path: str) -> None:
    if not os.path.exists(file_path):
        raise PDFLoadError(f"File not found: {file_path}")

    if not file_path.lower().endswith(".pdf"):
        raise PDFLoadError("Only PDF files are supported.")

    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    if size_mb > MAX_FILE_SIZE_MB:
        raise PDFLoadError(
            f"File size {size_mb:.2f} MB exceeds the {MAX_FILE_SIZE_MB} MB limit."
        )

    try:
        with pdfplumber.open(file_path) as pdf:
            if len(pdf.pages) == 0:
                raise PDFLoadError("PDF has no readable pages.")
    except Exception as exc:
        raise PDFLoadError(f"Invalid or corrupted PDF: {exc}")


def clean_text(raw_text: str) -> str:
    if not raw_text:
        return ""
    text = re.sub(r"[ \t]+", " ", raw_text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)
    return text.strip()


def extract_pages(file_path: str) -> list:
    """
    Returns a list of dicts: [{"page_number": int, "text": str, "document_name": str}, ...]
    Only non-empty pages are returned.
    """
    validate_pdf(file_path)
    document_name = os.path.basename(file_path)
    pages = []

    with pdfplumber.open(file_path) as pdf:
        for i, page in enumerate(pdf.pages, start=1):
            raw = page.extract_text() or ""
            cleaned = clean_text(raw)
            if cleaned:
                pages.append(
                    {
                        "page_number": i,
                        "text": cleaned,
                        "document_name": document_name,
                    }
                )

    if not pages:
        raise PDFLoadError("No extractable text found in PDF (it may be scanned/image-only).")

    return pages


def extract_multiple(file_paths: list) -> list:
    """
    Extracts pages from multiple PDFs. Skips files that fail, collecting errors.
    Returns (all_pages, errors)
    """
    all_pages = []
    errors = []
    for path in file_paths:
        try:
            all_pages.extend(extract_pages(path))
        except PDFLoadError as exc:
            errors.append({"file": os.path.basename(path), "error": str(exc)})
    return all_pages, errors


In [ ]:
%%writefile text_chunker.py
"""
text_chunker.py
Splits page-wise text into overlapping chunks with adjustable size,
while preserving page number and document name metadata per chunk.
"""

try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter


def chunk_pages(pages: list, chunk_size: int = 500, chunk_overlap: int = 80) -> list:
    """
    pages: list of {"page_number": int, "text": str, "document_name": str}
    Returns: list of {"chunk_id": str, "text": str, "page_number": int, "document_name": str}
    """
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive.")
    if chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap must be smaller than chunk_size.")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    chunks = []
    chunk_counter = 0

    for page in pages:
        page_chunks = splitter.split_text(page["text"])
        for piece in page_chunks:
            piece = piece.strip()
            if not piece:
                continue
            chunk_counter += 1
            doc_tag = page["document_name"].replace(" ", "_").replace(".pdf", "")
            chunk_id = f"{doc_tag}_p{page['page_number']}_c{chunk_counter}"
            chunks.append(
                {
                    "chunk_id": chunk_id,
                    "text": piece,
                    "page_number": page["page_number"],
                    "document_name": page["document_name"],
                }
            )

    if not chunks:
        raise ValueError("Chunking produced no chunks. Check input text.")

    return chunks


In [ ]:
%%writefile embedder.py
"""
embedder.py
Wraps a Sentence-Transformer model to convert text chunks into vector embeddings.
Using all-MiniLM-L6-v2: 384-dim, lightweight, runs comfortably on Colab free tier.
"""

from sentence_transformers import SentenceTransformer


EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
EMBEDDING_DIMENSION = 384


class Embedder:
    def __init__(self, model_name: str = EMBEDDING_MODEL_NAME):
        self.model = SentenceTransformer(model_name)
        self.dimension = self.model.get_sentence_embedding_dimension()

    def embed_texts(self, texts: list) -> list:
        if not texts:
            raise ValueError("No texts provided for embedding.")
        embeddings = self.model.encode(
            texts, show_progress_bar=False, convert_to_numpy=True
        )
        return embeddings.tolist()

    def embed_query(self, query: str) -> list:
        if not query or not query.strip():
            raise ValueError("Query text is empty.")
        return self.model.encode([query], convert_to_numpy=True)[0].tolist()

In [ ]:
%%writefile vector_store.py
"""
vector_store.py
Handles Pinecone index creation, namespace usage, upserting, querying,
and metadata-based filtering.
"""

import time
from pinecone import Pinecone, ServerlessSpec


class VectorStoreError(Exception):
    pass


class PineconeVectorStore:
    def __init__(
        self,
        api_key: str,
        index_name: str,
        dimension: int,
        namespace: str = "rag-default",
        cloud: str = "aws",
        region: str = "us-east-1",
    ):
        if not api_key:
            raise VectorStoreError("Pinecone API key is missing.")

        self.namespace = namespace
        self.index_name = index_name

        try:
            self.pc = Pinecone(api_key=api_key)
        except Exception as exc:
            raise VectorStoreError(f"Failed to connect to Pinecone: {exc}")

        try:
            existing_indexes = [idx["name"] for idx in self.pc.list_indexes()]
            if index_name not in existing_indexes:
                self.pc.create_index(
                    name=index_name,
                    dimension=dimension,
                    metric="cosine",
                    spec=ServerlessSpec(cloud=cloud, region=region),
                )
                while not self.pc.describe_index(index_name).status["ready"]:
                    time.sleep(1)

            self.index = self.pc.Index(index_name)
        except Exception as exc:
            raise VectorStoreError(f"Failed to create/connect index: {exc}")

    def upsert_chunks(self, chunks: list, embeddings: list, batch_size: int = 100) -> int:
        if len(chunks) != len(embeddings):
            raise VectorStoreError("Chunks and embeddings count mismatch.")

        vectors = []
        for chunk, vector in zip(chunks, embeddings):
            vectors.append(
                {
                    "id": chunk["chunk_id"],
                    "values": vector,
                    "metadata": {
                        "text": chunk["text"],
                        "page_number": chunk["page_number"],
                        "document_name": chunk["document_name"],
                    },
                }
            )

        upserted = 0
        try:
            for i in range(0, len(vectors), batch_size):
                batch = vectors[i : i + batch_size]
                self.index.upsert(vectors=batch, namespace=self.namespace)
                upserted += len(batch)
        except Exception as exc:
            raise VectorStoreError(f"Upsert failed: {exc}")

        return upserted

    def query(
        self,
        query_vector: list,
        top_k: int = 5,
        page_filter: int = None,
        document_filter: str = None,
    ) -> list:
        filter_dict = {}
        if page_filter is not None:
            filter_dict["page_number"] = {"$eq": page_filter}
        if document_filter:
            filter_dict["document_name"] = {"$eq": document_filter}

        try:
            result = self.index.query(
                vector=query_vector,
                top_k=top_k,
                namespace=self.namespace,
                include_metadata=True,
                filter=filter_dict if filter_dict else None,
            )
        except Exception as exc:
            raise VectorStoreError(f"Pinecone query failed: {exc}")

        matches = []
        for match in result.get("matches", []):
            matches.append(
                {
                    "score": match["score"],
                    "text": match["metadata"].get("text", ""),
                    "page_number": match["metadata"].get("page_number"),
                    "document_name": match["metadata"].get("document_name"),
                }
            )
        return matches

    def delete_namespace(self) -> None:
        try:
            self.index.delete(delete_all=True, namespace=self.namespace)
        except Exception:
            pass


In [ ]:
%%writefile retriever.py
"""
retriever.py
Retrieves top-k relevant chunks for a query, applies similarity threshold,
and computes a simple confidence score.
"""

from embedder import Embedder
from vector_store import PineconeVectorStore, VectorStoreError


class RetrieverError(Exception):
    pass


class Retriever:
    def __init__(self, embedder: Embedder, vector_store: PineconeVectorStore):
        self.embedder = embedder
        self.vector_store = vector_store

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        similarity_threshold: float = 0.3,
        page_filter: int = None,
        document_filter: str = None,
    ) -> dict:
        if not query or not query.strip():
            raise RetrieverError("Query cannot be empty.")

        try:
            query_vector = self.embedder.embed_query(query)
            raw_matches = self.vector_store.query(
                query_vector=query_vector,
                top_k=top_k,
                page_filter=page_filter,
                document_filter=document_filter,
            )
        except VectorStoreError as exc:
            raise RetrieverError(str(exc))

        filtered = [m for m in raw_matches if m["score"] >= similarity_threshold]

        if filtered:
            confidence = round(sum(m["score"] for m in filtered) / len(filtered), 4)
        else:
            confidence = 0.0

        return {
            "matches": filtered,
            "confidence": confidence,
            "has_context": len(filtered) > 0,
        }


In [ ]:
%%writefile generator.py
"""
generator.py
Generates answers strictly from retrieved context using the Groq LLM API
(Llama 3). Refuses to answer when context is insufficient, to prevent
hallucination.
"""

from groq import Groq


NO_ANSWER_MESSAGE = "The answer is not available in the provided document."

SYSTEM_PROMPT = (
    "You are a document question-answering assistant. "
    "You must answer strictly using only the provided context excerpts. "
    "Do not use any outside knowledge. Do not guess or infer beyond the context. "
    f"If the context does not contain enough information to answer, "
    f'respond with exactly: "{NO_ANSWER_MESSAGE}"'
)


class GeneratorError(Exception):
    pass


class AnswerGenerator:
    def __init__(self, api_key: str, model: str = "llama-3.1-8b-instant"):
        if not api_key:
            raise GeneratorError("Groq API key is missing.")
        self.client = Groq(api_key=api_key)
        self.model = model

    def _build_context(self, matches: list) -> str:
        blocks = []
        for i, m in enumerate(matches, start=1):
            blocks.append(
                f"[Excerpt {i} | {m['document_name']} | Page {m['page_number']}]\n{m['text']}"
            )
        return "\n\n".join(blocks)

    def generate_answer(self, query: str, retrieval_result: dict) -> dict:
        if not retrieval_result["has_context"]:
            return {
                "answer": NO_ANSWER_MESSAGE,
                "sources": [],
                "confidence": 0.0,
            }

        context = self._build_context(retrieval_result["matches"])
        user_prompt = (
            f"Context:\n{context}\n\n"
            f"Question: {query}\n\n"
            "Answer using only the context above. Include no outside information."
        )

        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0.1,
                max_tokens=600,
            )
            answer_text = response.choices[0].message.content.strip()
        except Exception as exc:
            raise GeneratorError(f"LLM generation failed: {exc}")

        sources = [
            {
                "document_name": m["document_name"],
                "page_number": m["page_number"],
                "excerpt": m["text"][:250] + ("..." if len(m["text"]) > 250 else ""),
                "similarity_score": round(m["score"], 4),
            }
            for m in retrieval_result["matches"]
        ]

        return {
            "answer": answer_text,
            "sources": sources,
            "confidence": retrieval_result["confidence"],
        }


In [ ]:
%%writefile query_logger.py
"""
query_logger.py
Logs user queries to a local file and maintains in-session query history.
"""

import csv
import os
from datetime import datetime


LOG_FILE = "query_log.csv"


def log_query(query: str, confidence: float, num_sources: int) -> None:
    file_exists = os.path.exists(LOG_FILE)
    with open(LOG_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["timestamp", "query", "confidence", "num_sources"])
        writer.writerow(
            [datetime.now().isoformat(timespec="seconds"), query, confidence, num_sources]
        )


def read_log(limit: int = 20) -> list:
    if not os.path.exists(LOG_FILE):
        return []
    with open(LOG_FILE, "r", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    return rows[-limit:]


In [ ]:
%%writefile rag_pipeline.py
"""
rag_pipeline.py
Orchestrates the full pipeline: loading, chunking, embedding, indexing,
retrieval, and generation. Acts as the single entry point for the app.
"""

from pdf_loader import extract_multiple, PDFLoadError
from text_chunker import chunk_pages
from embedder import Embedder, EMBEDDING_DIMENSION
from vector_store import PineconeVectorStore, VectorStoreError
from retriever import Retriever, RetrieverError
from generator import AnswerGenerator, GeneratorError
from query_logger import log_query


class RAGPipeline:
    def __init__(self, pinecone_api_key: str, groq_api_key: str, index_name: str, namespace: str):
        self.embedder = Embedder()
        self.vector_store = PineconeVectorStore(
            api_key=pinecone_api_key,
            index_name=index_name,
            dimension=EMBEDDING_DIMENSION,
            namespace=namespace,
        )
        self.retriever = Retriever(self.embedder, self.vector_store)
        self.generator = AnswerGenerator(api_key=groq_api_key)
        self.indexed_documents = []

    def ingest(self, file_paths: list, chunk_size: int = 500, chunk_overlap: int = 80) -> dict:
        pages, errors = extract_multiple(file_paths)
        if not pages:
            raise PDFLoadError("No valid PDF content could be extracted. " + str(errors))

        chunks = chunk_pages(pages, chunk_size=chunk_size, chunk_overlap=chunk_overlap)
        texts = [c["text"] for c in chunks]
        embeddings = self.embedder.embed_texts(texts)
        upserted_count = self.vector_store.upsert_chunks(chunks, embeddings)

        doc_names = sorted(set(c["document_name"] for c in chunks))
        self.indexed_documents = sorted(set(self.indexed_documents) | set(doc_names))

        return {
            "pages_processed": len(pages),
            "chunks_created": len(chunks),
            "vectors_upserted": upserted_count,
            "documents": doc_names,
            "errors": errors,
        }

    def ask(
        self,
        query: str,
        top_k: int = 5,
        similarity_threshold: float = 0.3,
        page_filter: int = None,
        document_filter: str = None,
    ) -> dict:
        try:
            retrieval_result = self.retriever.retrieve(
                query=query,
                top_k=top_k,
                similarity_threshold=similarity_threshold,
                page_filter=page_filter,
                document_filter=document_filter,
            )
            result = self.generator.generate_answer(query, retrieval_result)
        except (RetrieverError, GeneratorError) as exc:
            result = {"answer": f"Error: {exc}", "sources": [], "confidence": 0.0}

        log_query(query, result["confidence"], len(result["sources"]))
        return result


## 12. Verify Modules Import Correctly

In [ ]:
from pdf_loader import extract_multiple, PDFLoadError
from text_chunker import chunk_pages
from embedder import Embedder, EMBEDDING_DIMENSION
from vector_store import PineconeVectorStore, VectorStoreError
from retriever import Retriever, RetrieverError
from generator import AnswerGenerator, GeneratorError
from rag_pipeline import RAGPipeline

print("All modules imported successfully.")

## 13. Upload PDF Document(s)

Upload at least one PDF file (max 20 MB each). Multiple files can be selected for multi-document support.

In [ ]:
from google.colab import files
import os

uploaded = files.upload()
pdf_paths = []
for fname in uploaded.keys():
    size_mb = os.path.getsize(fname) / (1024 * 1024)
    print(f"{fname}: {size_mb:.2f} MB")
    if size_mb > 20:
        print(f"WARNING: {fname} exceeds 20 MB and will be rejected during processing.")
    pdf_paths.append(fname)

print("\nUploaded files:", pdf_paths)

## 14. Initialize the RAG Pipeline

This connects to Pinecone (creating the index if it does not exist), loads the embedding model, and sets up the Groq-based generator.

In [ ]:
from rag_pipeline import RAGPipeline

INDEX_NAME = "rag-intermediate-index"
NAMESPACE = "colab-session"

pipeline = RAGPipeline(
    pinecone_api_key=os.environ["PINECONE_API_KEY"],
    groq_api_key=os.environ["GROQ_API_KEY"],
    index_name=INDEX_NAME,
    namespace=NAMESPACE,
)

print("Pipeline initialized. Connected to Pinecone index:", INDEX_NAME)

In [ ]:
pipeline.vector_store.delete_namespace()
print("Namespace cleared. Ready for a fresh, duplicate-free ingestion of the new PDF.")

## 15. Process and Index the Document(s)

Adjust `CHUNK_SIZE` and `CHUNK_OVERLAP` below to control chunking granularity (this maps to the "adjustable chunk size" requirement).

In [ ]:
CHUNK_SIZE = 500
CHUNK_OVERLAP = 80

ingest_result = pipeline.ingest(pdf_paths, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

print("Pages processed:", ingest_result["pages_processed"])
print("Chunks created:", ingest_result["chunks_created"])
print("Vectors upserted into Pinecone:", ingest_result["vectors_upserted"])
print("Indexed documents:", ingest_result["documents"])
if ingest_result["errors"]:
    print("Errors encountered:", ingest_result["errors"])

## 16. Testing — Sample Queries and Expected Outputs

This section demonstrates every core feature: retrieval, source attribution, confidence scoring, adjustable top-k, metadata filtering, and hallucination prevention (out-of-context query).

**Sample Input 1:** A question that IS answerable from the document.
**Expected Output:** A grounded answer plus page number(s), excerpt(s), and similarity score(s).

**Sample Input 2:** A question with adjustable `top_k` and metadata filtering by page number.

In [ ]:
TOP_K = 3
SIMILARITY_THRESHOLD = 0.3

sample_query_1 = "What are the three types of machine learning?"

result_1 = pipeline.ask(
    query=sample_query_1,
    top_k=TOP_K,
    similarity_threshold=SIMILARITY_THRESHOLD,
)

print("Question:", sample_query_1)
print("\nAnswer:", result_1["answer"])
print("\nConfidence Score:", result_1["confidence"])
print("\nSources:")
for src in result_1["sources"]:
    print(f"  - {src['document_name']} | Page {src['page_number']} | Score {src['similarity_score']}")
    print(f"    Excerpt: {src['excerpt']}")

In [ ]:
sample_query_2 = "What is Retrieval-Augmented Generation and how does it reduce hallucination?"
PAGE_TO_FILTER = 3

result_2 = pipeline.ask(
    query=sample_query_2,
    top_k=3,
    similarity_threshold=0.2,
    page_filter=PAGE_TO_FILTER,
)

print("Question:", sample_query_2, f"(filtered to page {PAGE_TO_FILTER})")
print("\nAnswer:", result_2["answer"])
print("\nConfidence Score:", result_2["confidence"])
for src in result_2["sources"]:
    print(f"  - Page {src['page_number']} | Score {src['similarity_score']}")

**Sample Input 3 — Hallucination Prevention Test:** A question deliberately unrelated to the document content.
**Expected Output:** The system should respond that the answer is not available, instead of guessing.

In [ ]:
sample_query_3 = "What is the boiling point of water?"

result_3 = pipeline.ask(
    query=sample_query_3,
    top_k=TOP_K,
    similarity_threshold=0.5,
)

print("Question:", sample_query_3)
print("\nAnswer:", result_3["answer"])
print("\nConfidence Score:", result_3["confidence"])
print("\n(Expected: 'The answer is not available in the provided document.' or a low-confidence, empty-source result)")

**Query History Check** — confirms session logging is working.

In [ ]:
from query_logger import read_log

print("Recent query log:")
for row in read_log(limit=10):
    print(row)

In [ ]:
%%writefile app.py
"""
app.py
Streamlit interface for the Intermediate RAG System.
Provides PDF upload, adjustable chunking/retrieval controls, metadata
filtering, confidence display, and query history.
"""

import os
import streamlit as st
from rag_pipeline import RAGPipeline
from pdf_loader import PDFLoadError
from query_logger import read_log

st.set_page_config(page_title="Intermediate RAG System", layout="wide", initial_sidebar_state="expanded")

# ---------------------------------------------------------------------------
# Theme: "Reading Room" — warm paper background, ink-navy sidebar, amber
# highlighter accents (evokes marking up a physical document), teal for
# grounded/high-confidence results. Fraunces for headings, Inter for body.
# ---------------------------------------------------------------------------
CUSTOM_CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Fraunces:opsz,wght@9..144,400;9..144,600;9..144,700&family=Inter:wght@400;500;600;700&display=swap');

:root {
    --ink: #1B2A4A;
    --ink-soft: #4A5568;
    --paper: #FBF8F1;
    --paper-raised: #FFFFFF;
    --amber: #E3A008;
    --amber-soft: #FDF0D0;
    --teal: #2F8F82;
    --teal-soft: #E3F2EF;
    --coral: #C1502E;
    --coral-soft: #FBEAE3;
    --border: #E7DFCE;
}

html, body, [class*="css"] {
    font-family: 'Inter', sans-serif;
}

.stApp {
    background: var(--paper);
}

h1, h2, h3 {
    font-family: 'Fraunces', serif !important;
    color: var(--ink) !important;
    font-weight: 600 !important;
}

.hero-wrap {
    padding: 6px 0 18px 0;
    margin-bottom: 8px;
    border-bottom: 1px solid var(--border);
}
.hero-title {
    font-family: 'Fraunces', serif;
    font-size: 2.3rem;
    font-weight: 700;
    color: var(--ink);
    line-height: 1.2;
    margin: 0;
}
.hero-caption {
    font-family: 'Inter', sans-serif;
    color: var(--ink-soft);
    font-size: 0.98rem;
    margin-top: 6px;
}

.section-header {
    font-family: 'Fraunces', serif;
    font-size: 1.35rem;
    font-weight: 600;
    color: var(--ink);
    margin: 6px 0 2px 0;
    padding-bottom: 8px;
    border-bottom: 3px solid var(--amber);
    display: inline-block;
}
.section-wrap { margin-top: 28px; margin-bottom: 6px; }

/* Sidebar background */
section[data-testid="stSidebar"] {
    background: var(--ink);
}
section[data-testid="stSidebar"] label,
section[data-testid="stSidebar"] p,
section[data-testid="stSidebar"] span,
section[data-testid="stSidebar"] h1,
section[data-testid="stSidebar"] h2,
section[data-testid="stSidebar"] h3 {
    color: #F0EFE8 !important;
}

/* Sidebar text inputs — fixes white-on-white invisible text */
section[data-testid="stSidebar"] div[data-baseweb="input"],
section[data-testid="stSidebar"] div[data-baseweb="base-input"] {
    background-color: rgba(255,255,255,0.12) !important;
    border-radius: 8px !important;
}
section[data-testid="stSidebar"] input[type="text"],
section[data-testid="stSidebar"] input[type="password"] {
    background-color: transparent !important;
    color: #FFFFFF !important;
    caret-color: #FFFFFF !important;
}
section[data-testid="stSidebar"] input::placeholder {
    color: rgba(255,255,255,0.55) !important;
}
section[data-testid="stSidebar"] div[data-baseweb="select"] > div {
    background-color: rgba(255,255,255,0.12) !important;
    color: #FFFFFF !important;
    border-radius: 8px !important;
}
section[data-testid="stSidebar"] div[data-baseweb="select"] span {
    color: #FFFFFF !important;
}
section[data-testid="stSidebar"] svg {
    fill: #FFFFFF !important;
}

section[data-testid="stSidebar"] .stSlider [data-baseweb="slider"] div[role="slider"] {
    background-color: var(--amber) !important;
}
section[data-testid="stSidebar"] hr {
    border-color: rgba(255,255,255,0.15) !important;
}

.stButton > button {
    background: var(--amber) !important;
    color: var(--ink) !important;
    border: none !important;
    border-radius: 8px !important;
    font-weight: 600 !important;
    padding: 0.55em 1.2em !important;
    transition: transform 0.08s ease, box-shadow 0.15s ease;
    box-shadow: 0 2px 0 rgba(27, 42, 74, 0.15);
}
.stButton > button:hover {
    transform: translateY(-1px);
    box-shadow: 0 4px 10px rgba(227, 160, 8, 0.35);
    color: var(--ink) !important;
}
section[data-testid="stSidebar"] .stButton > button {
    background: var(--teal) !important;
    color: #FFFFFF !important;
    width: 100%;
}

[data-testid="stFileUploaderDropzone"] {
    background: var(--paper-raised) !important;
    border: 1.5px dashed var(--border) !important;
    border-radius: 10px !important;
}

div[data-testid="stExpander"] {
    background: var(--paper-raised);
    border: 1px solid var(--border) !important;
    border-left: 4px solid var(--teal) !important;
    border-radius: 8px !important;
    margin-bottom: 8px;
}

div[data-testid="stAlertContentSuccess"] { color: #1E5B4F !important; }
div[data-testid="stAlertContentError"] { color: #8C2E15 !important; }

.conf-badge {
    display: inline-flex;
    align-items: center;
    gap: 8px;
    padding: 6px 14px;
    border-radius: 999px;
    font-family: 'Inter', sans-serif;
    font-weight: 600;
    font-size: 0.92rem;
    margin: 6px 0 14px 0;
}
.conf-dot { width: 9px; height: 9px; border-radius: 50%; }
.conf-high { background: var(--teal-soft); color: #1E5B4F; }
.conf-high .conf-dot { background: var(--teal); }
.conf-mid { background: var(--amber-soft); color: #7A5900; }
.conf-mid .conf-dot { background: var(--amber); }
.conf-low { background: var(--coral-soft); color: #8C2E15; }
.conf-low .conf-dot { background: var(--coral); }

.answer-card {
    background: var(--paper-raised);
    border: 1px solid var(--border);
    border-left: 5px solid var(--ink);
    border-radius: 10px;
    padding: 18px 20px;
    margin-bottom: 6px;
    font-size: 1.02rem;
    color: var(--ink);
    line-height: 1.55;
}

.qa-block {
    background: var(--paper-raised);
    border: 1px solid var(--border);
    border-radius: 10px;
    padding: 12px 16px;
    margin-bottom: 10px;
}
.qa-q { color: var(--ink); font-weight: 600; margin-bottom: 4px; }
.qa-a { color: var(--ink-soft); }
</style>
"""

st.markdown(CUSTOM_CSS, unsafe_allow_html=True)

st.markdown(
    """
    <div class="hero-wrap">
        <div class="hero-title">Intermediate RAG System — PDF Question Answering</div>
        <div class="hero-caption">Retrieval-Augmented Generation using Pinecone, Sentence-Transformers, and Groq LLM</div>
    </div>
    """,
    unsafe_allow_html=True,
)


def confidence_badge(score: float) -> str:
    if score >= 0.6:
        tier, label = "conf-high", "Strong grounding"
    elif score > 0.0:
        tier, label = "conf-mid", "Partial grounding"
    else:
        tier, label = "conf-low", "No grounding found"
    return (
        f'<div class="conf-badge {tier}"><span class="conf-dot"></span>'
        f'Confidence {score:.2f} · {label}</div>'
    )


if "pipeline" not in st.session_state:
    st.session_state.pipeline = None
if "history" not in st.session_state:
    st.session_state.history = []
if "indexed_docs" not in st.session_state:
    st.session_state.indexed_docs = []

with st.sidebar:
    st.header("Configuration")

    pinecone_key = st.text_input("Pinecone API Key", type="password", value=os.environ.get("PINECONE_API_KEY", ""))
    groq_key = st.text_input("Groq API Key", type="password", value=os.environ.get("GROQ_API_KEY", ""))
    index_name = st.text_input("Pinecone Index Name", value="rag-intermediate-index")
    # Default namespace matches the Colab notebook's NAMESPACE ("colab-session"),
    # so Streamlit reads the same indexed data and produces the same confidence
    # scores as the notebook testing cells, instead of a separate empty/duplicate namespace.
    namespace = st.text_input("Namespace", value="colab-session")

    st.divider()
    st.subheader("Chunking Settings")
    chunk_size = st.slider("Chunk Size (characters)", 200, 1500, 500, step=50)
    chunk_overlap = st.slider("Chunk Overlap", 0, 300, 80, step=10)

    st.divider()
    st.subheader("Retrieval Settings")
    top_k = st.slider("Top-K Results", 1, 15, 5)
    similarity_threshold = st.slider("Similarity Threshold", 0.0, 1.0, 0.3, step=0.05)
    page_filter_input = st.text_input("Filter by Page Number (optional)")
    document_filter_input = st.selectbox(
        "Filter by Document (optional)",
        options=["All"] + st.session_state.indexed_docs,
    )

    st.divider()
    if st.button("Initialize / Connect Pipeline", use_container_width=True):
        try:
            if not pinecone_key or not groq_key:
                st.error("Both Pinecone and Groq API keys are required.")
            else:
                st.session_state.pipeline = RAGPipeline(
                    pinecone_api_key=pinecone_key,
                    groq_api_key=groq_key,
                    index_name=index_name,
                    namespace=namespace,
                )
                st.success("Pipeline connected successfully.")
        except Exception as exc:
            st.error(f"Connection failed: {exc}")

st.markdown('<div class="section-wrap"><span class="section-header">1. Upload PDF Document(s)</span></div>', unsafe_allow_html=True)
uploaded_files = st.file_uploader(
    "Upload one or more PDF files (max 20 MB each)",
    type=["pdf"],
    accept_multiple_files=True,
    label_visibility="collapsed",
)

if st.button("Process and Index Documents"):
    if st.session_state.pipeline is None:
        st.error("Please initialize the pipeline first using the sidebar.")
    elif not uploaded_files:
        st.error("Please upload at least one PDF file.")
    else:
        temp_paths = []
        for uf in uploaded_files:
            temp_path = os.path.join("/tmp", uf.name)
            with open(temp_path, "wb") as f:
                f.write(uf.getbuffer())
            temp_paths.append(temp_path)

        try:
            with st.spinner("Extracting, chunking, embedding, and indexing..."):
                result = st.session_state.pipeline.ingest(
                    temp_paths, chunk_size=chunk_size, chunk_overlap=chunk_overlap
                )
            st.session_state.indexed_docs = st.session_state.pipeline.indexed_documents
            st.success(
                f"Indexed {result['chunks_created']} chunks from "
                f"{result['pages_processed']} pages across {len(result['documents'])} document(s)."
            )
            if result["errors"]:
                st.warning(f"Some files failed: {result['errors']}")
        except PDFLoadError as exc:
            st.error(f"PDF processing error: {exc}")
        except Exception as exc:
            st.error(f"Unexpected error during indexing: {exc}")

st.markdown('<div class="section-wrap"><span class="section-header">2. Ask a Question</span></div>', unsafe_allow_html=True)
query = st.text_input("Enter your question about the uploaded document(s)", label_visibility="collapsed", placeholder="Ask something about your document...")

if st.button("Get Answer"):
    if st.session_state.pipeline is None:
        st.error("Please initialize the pipeline first using the sidebar.")
    elif not query or not query.strip():
        st.error("Query cannot be empty.")
    else:
        page_filter = int(page_filter_input) if page_filter_input.strip().isdigit() else None
        document_filter = None if document_filter_input == "All" else document_filter_input

        with st.spinner("Retrieving context and generating answer..."):
            result = st.session_state.pipeline.ask(
                query=query,
                top_k=top_k,
                similarity_threshold=similarity_threshold,
                page_filter=page_filter,
                document_filter=document_filter,
            )

        st.session_state.history.append({"query": query, "answer": result["answer"]})

        st.markdown(f'<div class="answer-card">{result["answer"]}</div>', unsafe_allow_html=True)
        st.markdown(confidence_badge(result["confidence"]), unsafe_allow_html=True)

        if result["sources"]:
            st.markdown("**Source Attribution**")
            for i, src in enumerate(result["sources"], start=1):
                with st.expander(f"Source {i} · {src['document_name']} · Page {src['page_number']} · score {src['similarity_score']}"):
                    st.write(src["excerpt"])

st.markdown('<div class="section-wrap"><span class="section-header">3. Query History (This Session)</span></div>', unsafe_allow_html=True)
if st.session_state.history:
    for item in reversed(st.session_state.history[-10:]):
        st.markdown(
            f'<div class="qa-block"><div class="qa-q">Q: {item["query"]}</div>'
            f'<div class="qa-a">A: {item["answer"]}</div></div>',
            unsafe_allow_html=True,
        )
else:
    st.info("No queries yet in this session.")

with st.expander("View Query Log (persisted)"):
    log_rows = read_log(limit=20)
    if log_rows:
        st.table(log_rows)
    else:
        st.write("No logged queries yet.")


## 17. Launch the Streamlit Web Interface

This runs the full interactive app (upload, adjustable chunking/retrieval controls, metadata filtering, confidence display, source attribution, and query history) and exposes it via a public ngrok URL.

Get a free ngrok authtoken at https://dashboard.ngrok.com/get-started/your-authtoken (required once per session).

In [ ]:
from pyngrok import ngrok

NGROK_AUTHTOKEN = getpass("Enter your ngrok authtoken: ")
ngrok.set_auth_token(NGROK_AUTHTOKEN)

ngrok.kill()
public_url = ngrok.connect(8501)
print("Streamlit app will be available at:", public_url)

In [ ]:
!streamlit run app.py --server.port 8501 --server.headless true &>/content/streamlit_log.txt &
import time
time.sleep(8)
print("Streamlit server started. Open the ngrok URL printed above.")

## 18. Stopping the App

Run this cell when you are done to free the port and close the tunnel.

In [ ]:
!kill -9 $(lsof -t -i:8501) 2>/dev/null
ngrok.kill()
print("Streamlit app stopped and tunnel closed.")

## 19. Notes, Limitations, and Enhancements Implemented

**Mandatory intermediate enhancements implemented (5 of 7):**
1. Multi-document support (upload and query across multiple PDFs)
2. Adjustable chunk size and overlap from the UI
3. Adjustable top-k retrieval from the UI
4. Metadata filtering by page number (and by document)
5. Confidence scoring displayed with every answer
6. Query history (in-session) — bonus
7. Persistent query logging to `query_log.csv` — bonus

**Known limitations:**
- Scanned/image-only PDFs are not supported (no OCR layer); text-based PDFs only.
- Answer quality depends on Groq model availability and rate limits on the free tier.
- Pinecone free tier has index/storage limits; very large documents may require plan upgrades.

**Possible future improvements:**
- Add OCR (e.g., Tesseract) for scanned documents.
- Add hybrid search (keyword + semantic) for improved retrieval accuracy.
- Add persistent chat-style conversation memory across sessions.
- Add automatic evaluation metrics (e.g., answer relevance scoring).
